<a href="https://colab.research.google.com/github/Pimentaah/Projeto-Transporte/blob/desenvolvimento/notebook/note_principal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Processo de Imigração para o Google Colab**

In [ ]:
!git clone -b desenvolvimento https://github.com/Pimentaah/Projeto-Transporte.git

In [ ]:
import os
!pip install python-dotenv

os.chdir('/content/Projeto-Transporte')

In [ ]:
os.chdir('/content/Projeto-Transporte')

print(os.getcwd())

**Consultando se estou na Branch certa para voltar a trabalhar**

In [ ]:
!git branch --show-current

**Importando Dados GTFS para o Colab, fiz o pull para atualizar.**

In [ ]:
%cd /content/Projeto-Transporte

!git pull origin desenvolvimento

In [ ]:
import os

PASTA_GTFS = "data/_sptrans_gtfs"

print("Arquivos GTFS encontrados:")
print(os.listdir(PASTA_GTFS))

# 1. Configuração e conexão com a API Olho Vivo

Nesta etapa, realizo a configuração inicial do projeto, importei as bibliotecas que posteriomente irei precisar e estabeleço a conexão com a API Olho Vivo da SPTrans.

In [ ]:
import os
import requests
import pandas as pd
from google.colab import userdata

TOKEN = userdata.get('TOKEN') # Guardando o Token salvo na KEY do Colab.

BASE_URL = "https://api.olhovivo.sptrans.com.br/v2.1" # Guardando a Url em uma variável.

print("Token encontrado:", TOKEN is not None) # Verificando se o Token existe

In [ ]:
sessao = requests.Session()

# Agora o notebook confere se o login foi aceito, em vez de só retornar a resposta.
try:
    resposta = sessao.post(
        f"{BASE_URL}/Login/Autenticar",
        params={"token": TOKEN},
        timeout=15
    )
    print("Código HTTP:", resposta.status_code)
    print("Resposta:", resposta.text)

    # A API responde o texto true quando o token é aceito.
    autenticado = resposta.status_code == 200 and resposta.text.strip().lower() == "true"
except requests.exceptions.RequestException as erro:
    print("Falha de conexão ao autenticar:", type(erro).__name__)
    autenticado = False

print("Autenticado na API Olho Vivo:", autenticado)
if not autenticado:
    print("ATENÇÃO: sem autenticação as consultas à API vão falhar. Confira o TOKEN nos Secrets do Colab.")

In [ ]:
resposta = sessao.get(f"{BASE_URL}/Linha/Buscar", params={"termosBusca": "8000"}, timeout=15) # Na variável resposta, vou pegar(Get) dentro da Base URl, o end-point
linhas = resposta.json()  # transforma o texto da resposta em lista do Python

tabela_linhas = pd.DataFrame(linhas) # Peguei a variavel LINHAS e transformei esse resultado em um DataFrame para se tornar uma tabela.
tabela_linhas

In [ ]:
import os

print("Diretório atual:")
print(os.getcwd())

print("\nConteúdo da pasta atual:")
print(os.listdir())

print("\nConteúdo da raiz do projeto:")
print(os.listdir("/content/Projeto-Transporte"))

In [ ]:
codigo_linha = 35274

resposta = sessao.get(f"{BASE_URL}/Parada/BuscarParadasPorLinha", # Chamando(Get) End-point da Base_URL para retornar informações de Paradas das Linhas.
                      params={"codigoLinha": codigo_linha}, timeout=15) # Passei como parametro que a resposta deve receber informações guardadas no código da variável codigo_linha.
paradas = pd.DataFrame(resposta.json()) # Estou transformando a resposta que veio em HTTPS em uma lista Json (Dicionário) e trasformando o Json em uma "DF" de Tabela, guardando em uma variável.

paradas = paradas.rename(columns={   # Utilizando o ".rename", eu tratei nomes das colunas para ficar mais prático de análisar olhando os nomes.
    "cp": "codigo_parada",
    "np": "nome_parada",
    "ed": "endereco",
    "py": "latitude",
    "px": "longitude"
})

print("Quantidade de paradas:", len(paradas)) #Linha Básica para verificar o tamanho da tabela paradas utilzando "Len"
paradas.head(15)  # Por tendência o ".head" pega as 5 primeiras linhas da tabela, mas podemos passar no parametro quantas linhas podemos ver.

# 2. Funções auxiliares para análise

    Nesta etapa, criei funções que serão usadas ao longo do projeto, principalmente para calcular distâncias geográficas e identificar paradas próximas de certos pontos.

In [ ]:
from math import radians, sin, cos, asin, sqrt # Importei da biblioteca Math, algumas operações matemáticas que vou precisar.

def distancia_m(lat1, lon1, lat2, lon2):
    """Distância em metros entre dois pontos (fórmula de Haversine)."""
    Raio = 6371000  # O numero atribuído a variável representa o raio da Terra em metros
    dlat = radians(lat2 - lat1) # Guarda o resultado passado para Radianos da conta de (Lat2 - Lat1) que está cotada em Graus.
    dlon = radians(lon2 - lon1) # Guarda o resultado passado para Radianos da conta de (Lon2 - Lon1) que está cotada em Graus.


    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * Raio * asin(sqrt(a))

In [ ]:

distancias = [] # Criando uma variável de lista para guardar as distancias
for i in range(1, len(paradas)): # Loop para pegar cada Parada da tabela Paradas
    anterior = paradas.iloc[i - 1] # Guardando o valor anterior do valor atual (i) do loop na variável "anterior".
    atual = paradas.iloc[i] # Guardando o valor atual do Loop (i) na variável "atual"
    distancias.append( # O append aqui serve para guardar na lista "distancia".
        distancia_m(anterior["latitude"], anterior["longitude"], # Dentro do append, chamei a função para calcular os parametros recebidos da latitude e longitude das variaveis "anterior" e "atual".
                    atual["latitude"], atual["longitude"])
    )

serie = pd.Series(distancias) # Guardei em uma variável e tranformei a lista "distancia" em uma Série com indices e valores para analisa melhor.
print(serie.describe()) # O describe serve para fazer uma análise Estatística sobre a Série "Distancias"
print("Saltos maiores que 1000 m:", (serie > 1000).sum(), "de", len(serie)) # Retornei utilizando um filtro, a contagem de distancias de são > 1000 e comparei com o tamanho da lista utilizando "Len".


In [ ]:
def paradas_proximas(tabela_paradas, lat, lon, raio_max): # Essa função serve para encontrar as paradas mais próximas do ponto de origem.
    """Devolve as paradas a até raio_max metros do ponto, da mais perto para a mais longe."""

    tabela = tabela_paradas.copy() # Copia o DataFrame original e salva em outra variável "teste", para não alterar o DataFrame Original.
    tabela["distancia_m"] = tabela.apply( # Estou criando uma coluna chamada "distancia_m" na nova tabela e estou aplicando uma Função com o apply.
        lambda parada: distancia_m(lat, lon, parada["latitude"], parada["longitude"]), axis=1 # Lamba é uma mini função dentro do apply que vai chamar a função distancia_m e passar os parametros das cordenadas do usuário e das paradas.
    ) # o "Axis = 1" significa que vai trabalhar o lambda pegando linha por linha, se fosse "0" seria para pegar de coluna em coluna

    return tabela[tabela["distancia_m"] <= raio_max].sort_values("distancia_m") # Aplicando um filtro para retornar uma tabela com apenas as distâncias que atendem o parametro de "Raio_max", e ordenando esses valores de forma ascentente.

In [ ]:
# Resolvi implementar uma função para validar todas as consultas à API, o objetivo é tratar os erros sempre do mesmo jeito
# exemplo de erros: falha de conexão, erro HTTP, resposta que não é JSON, mensagem de erro da API).

def consultar_api(endpoint, params):

    try:
        resposta = sessao.get(f"{BASE_URL}/{endpoint}", params=params, timeout=15)
    except requests.exceptions.RequestException:
        return None, "falha de conexão com a API Olho Vivo."

    if resposta.status_code == 401:
        return None, "a API não autorizou a consulta (HTTP 401). Rode de novo a célula de login da seção 1."
    if resposta.status_code != 200:
        return None, f"a API respondeu com erro HTTP {resposta.status_code}."

    try:
        dados = resposta.json()
    except ValueError:
        return None, "a API devolveu uma resposta que não está em formato JSON."

    if isinstance(dados, dict) and "Message" in dados:
        return None, "a API devolveu uma mensagem de erro em vez de dados."

    return dados, None

# 3. Importação e preparação dos dados GTFS

    Nesta etapa, importo os arquivos GTFS da SPTrans e verifico a estrutura das tabelas, seus registros e a integridade das informações.
    
Percebi que somente com as buscas da API Olho Vivo, estavam retornando informações do tráfego de forma desorganizada, para isso, resolvi buscar na mesma documentação do site uma forma de retornar essas buscas mais estruturadas, e encontrei o GTFS, que funciona como um guia oficial de como as linhas são desenhadas, Ou seja, enquanto a API retornava somente poucas linhas, ordem "aleatória" e sem mapa de trajeto, O GTFS vai nos complementar essa informação de forma coletiva com  a API Olho vivo.



In [ ]:
import os

PASTA_GTFS = "data/_sptrans_gtfs"

print(os.listdir(PASTA_GTFS))

In [ ]:
# Aqui eu estou carregando os arquivos GTFS

routes = pd.read_csv(f"{PASTA_GTFS}/routes.txt", dtype=str)
trips = pd.read_csv(f"{PASTA_GTFS}/trips.txt", dtype=str)
stops = pd.read_csv(f"{PASTA_GTFS}/stops.txt", dtype=str)
stop_times = pd.read_csv(f"{PASTA_GTFS}/stop_times.txt", dtype=str)

# Conferindo o conteúdo de cada tabela

for nome, tabela in [
    ("routes", routes),
    ("trips", trips),
    ("stops", stops),
    ("stop_times", stop_times)
]:
    print(f"\n{nome}")
    print("Quantidade de registros:", len(tabela))
    print("Colunas:", tabela.columns.tolist())


In [ ]:

# Verificando a estrutura e alguns registros das tabelas

print("ROTAS")
display(routes.head(3))

print("VIAGENS")
display(trips.head(3))

print("PARADAS")
display(stops.head(3))

print("SEQUÊNCIA DAS PARADAS")
display(stop_times.head(3))


In [ ]:

# Verificando os relacionamentos entre as tabelas GTFS:
# Estou verificando a inexistência dos requistos que pedi
# com o sinal de "~", esse sinal reprepresanta a inversão da verdade, ou seja, quero ver quantos numeros do que pedi
# estão sem um relacionamento, para isso utilizei a inversão de informação "~".

print("Viagens com route_id inexistente:")
print((~trips["route_id"].isin(routes["route_id"])).sum()) # Contando (.sum) se route_id está dentro de routes([route_id])  e invertendo com ~ para descobrir se existe algum relacionamento vazio

print("\nRegistros de stop_times com trip_id inexistente:")
print((~stop_times["trip_id"].isin(trips["trip_id"])).sum()) # Mesma relação com o comentário acima, porém com colunas diferentes.

print("\nRegistros de stop_times com stop_id inexistente:")
print((~stop_times["stop_id"].isin(stops["stop_id"])).sum()) # Mesma relação com o comentário acima, porém com colunas diferentes.


# 4. Relacionamento das tabelas GTFS

    Nesta etapa, utilizo relacionamentos entre as tabelas para reunir as informações de linhas, viagens e paradas em uma estrutura integrada.

Aqui eu inicio um processo interessante e importante para a conexão das informações (Tabelas), Nessa etapa do código, vou utilizar o conceito de Modelagem de Dados para relacionar as tabelas entre si.

In [ ]:
#11

# Fazendo o  1° relacionamento de viagens com suas respectivas linhas

# Aplicando o conceito de modelagem de dados - Left Join

viagens_linhas = trips.merge( # Merge - Faz a mistura das duas tabelas
    routes,
    on="route_id", # Informa qual coluna irá conectar as tabelas
    how="left" # Conecta as informações mas mantém todos os registros da tabela "trips" a esquerda.
)

display(viagens_linhas.head()) # Apresentando como ficou a relação com as tabelas que foram conectadas.


In [ ]:
# 2° Relacionamento

# Adicionando as paradas de cada viagem

viagens_paradas = viagens_linhas.merge( # Relacionando as paradas de cada viagem
    stop_times,
    on="trip_id", # Conectando as informações pela referência ("trip_id")
    how="left" # Mantendo as informações da tabela "viagens_linhas" a Esquerda
)

display(viagens_paradas.head())


In [ ]:
# 3° Ultimo relacionamento - Estrutura Geral GTFS

# Adicionando nome e coordenadas dos pontos

gtfs_completo = viagens_paradas.merge( # Essa varaiável irá guardar todas as informações das relações entre tabelas que fizemos.
    stops, # Tabela paradas quue irá se relacionar com a tabela acima.
    on="stop_id", # As informações vão se conectar pela coluna de referencia "stop_id"
    how="left" # A tabela viagens_paradas irá manter as informações a esquerda, enquanto a tabela stops ficará a direita de acordo com a relação.
)

display(gtfs_completo.head())


In [ ]:

# Conferindo a tabela integrada

print("Quantidade de registros:", len(gtfs_completo))
print("\nColunas disponíveis:")
print(gtfs_completo.columns.tolist())

print("\nInformações da tabela:")
gtfs_completo.info()


# 5. Análise exploratória das linhas e viagens

    Nesta etapa, realizo análises exploratórias dos dados GTFS para entender melhor a distribuição das linhas, a quantidade de viagens e a quantidade de paradas relacionadas a cada linha.

In [ ]:
# Seleciona o primeiro trip_id disponível
viagem_exemplo = gtfs_completo["trip_id"].iloc[0]

print("Viagem selecionada:", viagem_exemplo)

# Atribuíndo valor ás variáveis

paradas_viagem = gtfs_completo[ # Essa variável só irá guardar as informações do DataFrame gtfs_compelto
    gtfs_completo["trip_id"] == viagem_exemplo # Guarda todos os id_linha do DataFrame que for == "viagem exemplo"
].copy() # Copiando o DataFrame "gtfs_completo" com a filtragem para não alterar o DataFrame original

paradas_viagem["stop_sequence"] = pd.to_numeric( # Alterando as sequncia de parada para tipo numero
    paradas_viagem["stop_sequence"]
)

paradas_viagem = paradas_viagem.sort_values( # Ordenando os valores (stop_sequence) e ordem ascendente.
    "stop_sequence"
)

display( # Aqui estou mostrando quais informações (Colunas) quero que retorne na minha busca.
    paradas_viagem[
        [
            "trip_id",
            "stop_sequence",
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon"
        ]
    ]
)

In [ ]:
viagens_por_linha = ( # Guardando em uma variável a filtragem do algoritimo abaixo
    trips.groupby("route_id")["trip_id"] # Groupyby utilizado para agrupar as informações pelos identificadores passados. Trip_id é a coluna onde queremos analisar
    .nunique() # conta quantas viagem diferentes possui em cada linha
    .reset_index(name="quantidade_viagens") # Pega a filtragem e tranforma em uma tabela organizada, dando um nome para a coluna  a ser analisada.
)

display(viagens_por_linha.head(10))

In [ ]:
paradas_por_linha = (
    gtfs_completo.groupby("route_id")["stop_id"] # Agrupando pelo route_id dando como referência de busca a coluna "stop_id"
    .nunique() # Contando quantas paradas diferentes possuem em cada linha agrupada
    .reset_index(name="quantidade_paradas") # Ordenando os valores em uma tabela organizada e atribuindo um nome para a coluna
)

display(paradas_por_linha.head(10))

In [ ]:
resumo_linhas = ( # Fazendo um Merge de tabelas para resumir informações
    routes[["route_id", "route_short_name", "route_long_name"]] # Se referindo as colunas da tabela routes
    .merge(viagens_por_linha, on="route_id", how="left") # Relacionando os route_id que aparecem nas duas tabelas e relacionando essas informações mantendo a primeira tabela a esquerda.
    .merge(paradas_por_linha, on="route_id", how="left") # Faz um merge relacionado ao merge anterior e pega como referência o route_id para juntar as informações.
)

display(resumo_linhas.head(10))

In [ ]:
ranking_linhas = resumo_linhas.sort_values( # Variavel que irá rankear o algoritimo abaixo
    by="quantidade_viagens", # Selecionando a coluna ("quantidade_viagens") da variável resumo_linhas
    ascending=False # Ordenando de forma descentente as informações.
)

display(ranking_linhas.head(10)) # Retornando as 10 primeiras linhas da variável.

In [ ]:
linhas_sem_viagens = resumo_linhas[
    resumo_linhas["quantidade_viagens"].isna() # Buscando somente os registros na coluna quantidade_viagens que possuem valor nulo
]

display(linhas_sem_viagens) # Mostrando as linhas guardadas como nulas da busca acima

# 6. Identificação de paradas próximas à origem e ao destino

    Nesta etapa, utilizo coordenadas de origem e destino para identificar as paradas de transporte público próximas a esses pontos, considerando um raio de distância definido.

In [ ]:
# Buscando colunas de uma variável e guardando em outra variável para realizar uma analise sem alterar a variável com infos importantes.
paradas_gtfs = stops[
    ["stop_id", "stop_name", "stop_lat", "stop_lon"]
].copy()

#Alterando o nome das colunas para um nome intuítivo
paradas_gtfs = paradas_gtfs.rename(columns={
    "stop_lat": "latitude",
    "stop_lon": "longitude"
})

# Tranformando o tipo de dado das colunas e convertendo para Número
paradas_gtfs["latitude"] = pd.to_numeric(paradas_gtfs["latitude"])
paradas_gtfs["longitude"] = pd.to_numeric(paradas_gtfs["longitude"])

## 6.1 Definição das coordenadas de origem e destino

Neste bloco de código, defini as coordenadas geográficas de origem e destino e coloquei o raio de busca usado para localizar as paradas próximas a esses pontos.


In [ ]:
# Coordenadas de exemplo para o primeiro teste do algoritmo
origem = {
    "latitude": -23.5505,
    "longitude": -46.6333
}

destino = {
    "latitude": -23.5614,
    "longitude": -46.6559
}

raio_busca = 500

In [ ]:
# Chamando função que está recebendo parametros de Origem e guardando o resultado em uma variável
paradas_origem = paradas_proximas(
    paradas_gtfs,
    origem["latitude"],
    origem["longitude"],
    raio_busca
)

# Chamando função que está recebendo parametros de destino e guardando os resultados em uma variável
paradas_destino = paradas_proximas(
    paradas_gtfs,
    destino["latitude"],
    destino["longitude"],
    raio_busca
)

# Chamando as variáveis que guardaram os resultados da função
print("Paradas próximas da origem:", len(paradas_origem))
print("Paradas próximas do destino:", len(paradas_destino))

## 6.2 Identificação das linhas próximas à origem e ao destino
    
A partir das paradas encontradas, encontrei quais linhas atendem às regiões de origem e destino. Depois, verifico quais linhas aparecem nos dois conjuntos.

In [ ]:
# Bloco utilizado para identificar e guardar quais linhas passam próximas á origem que o passageiro está.
linhas_origem = set( # O set é usado para tranformar informações e um conjunto sem duplicadas, ou seja, não repetir as linhas retornadas.
    gtfs_completo[
        gtfs_completo["stop_id"].isin(paradas_origem["stop_id"]) # Verificando quais valores da tabela gtfs_completo está na tabela parada origem.
    ]["route_id"].unique() # Retornando valores unicos da coluna route_id.
)

# Bloco utilizado para verificar quais linhas passam próximas ao destino, sem haver linhas duplicadas.
linhas_destino = set(
    gtfs_completo[
        gtfs_completo["stop_id"].isin(paradas_destino["stop_id"])
    ]["route_id"].unique()
)

linhas_candidatas = linhas_origem.intersection(linhas_destino) # Verificando e guardando a intersseção de identificadores nas variáveis.

print("Linhas próximas da origem:", len(linhas_origem))
print("Linhas próximas do destino:", len(linhas_destino))
print("Linhas em comum:", len(linhas_candidatas))

## 7. Identificação de viagens diretas
Nesta etapa, relaciono as paradas próximas à origem e ao destino para identificar viagens que permitem fazer o percurso sem necessidade de troca de ônibus.

In [ ]:
# Identificando as paradas de embarque
embarques = gtfs_completo[
    gtfs_completo["stop_id"].isin(paradas_origem["stop_id"]) # Verificando quais valores da tabela gtfs_completo está na tabela parada origem.
][["route_id", "trip_id", "stop_id", "stop_sequence"]].copy() # Aqui eu estou filtrando quais são as colunas que me interessam e estou guardando em uma variável para não alterar o DF original.

# Identificando as paradas de desembarque
desembarques = gtfs_completo[
    gtfs_completo["stop_id"].isin(paradas_destino["stop_id"])
][["route_id", "trip_id", "stop_id", "stop_sequence"]].copy()

# Nesse bloco eu estou convertendo as colunas para número, futuramente será utilizada para comparações numéricas.
embarques["stop_sequence"] = pd.to_numeric(embarques["stop_sequence"])
desembarques["stop_sequence"] = pd.to_numeric(desembarques["stop_sequence"])

# Relacionando embarques e desembarques da mesma viagem
viagens_diretas = embarques.merge( # Estou relacioando as informações que se referem a mesma viagem.
    desembarques,
    on=["route_id", "trip_id"], # Nessa linha, mantive as informações que se referem a mesma viagem, no caso, os mesmo identidicadores de linha e viagem.
    suffixes=("_origem", "_destino") # Separei as informações de embarque e desembarque com sufixos para não confundir as informações que se referem a mesma viagem.
)

# Mantendo apenas viagens em que o destino vem depois da origem, ou seja, quero manter apenas viagens que o stop_sequence destino é maior que o de origem.
viagens_diretas = viagens_diretas[
    viagens_diretas["stop_sequence_destino"] > # Fazendo a comparação entre as colunas stop_sequence_destino e stop_sequence_origem, para manter apenas viagens que o destino vem depois da origem.
    viagens_diretas["stop_sequence_origem"]
]

display(viagens_diretas.head(10))

## 7.1 Complementação das informações de embarque e desembarque

Acrescento às opções de viagem os nomes das paradas de embarque e desembarque, suas respectivas distâncias e as informações das linhas.

In [ ]:
## Identificação de parada de embarque

# Nesse bloco estou fazendo um merge entre a tabela de viagens_diretas e a tabela paradas_origem, para trazer informações de nome da parada e distância da parada de embarque.
opcoes_viagem = viagens_diretas.merge(
    paradas_origem[["stop_id", "stop_name", "distancia_m"]],
    left_on="stop_id_origem", # o left on serve para relacionar qual coluna da tabela da esquerda vai se relacionar com a tabela da direita.
    right_on="stop_id", # O right on serve para relacionar qual informação da coluna da direita vai se relacionar com a coluna da esquerda.
    how="left" # Vou manter as informações da tabela da esquerda.
)

# Renomeando colunas para nomes mais entendiveis e intuitivos.
opcoes_viagem = opcoes_viagem.rename(columns={
    "stop_name": "parada_embarque",
    "distancia_m": "distancia_origem"
})

opcoes_viagem = opcoes_viagem.drop(columns="stop_id") # Apagando a coluna stop_id que não é mais útil.

display(opcoes_viagem.head())

In [ ]:
## Identificação de parada de desembarque

opcoes_viagem = opcoes_viagem.merge(
    paradas_destino[["stop_id", "stop_name", "distancia_m"]], # Filtrando as colunas que quero da tabela de destino.
    left_on="stop_id_destino", # Pegando a coluna que quero usar de referencia da tabela da esquerda.
    right_on="stop_id", # Pegando a coluna que quero usar de referencia da tabela da direita.
    how="left" # Vou manter as informações da tabela da esquerda.
)

opcoes_viagem = opcoes_viagem.rename(columns={ # Renomeando colunas para nomes mais entendiveis e intuitivos.
    "stop_name": "parada_desembarque",
    "distancia_m": "distancia_destino"
})

opcoes_viagem = opcoes_viagem.drop(columns="stop_id") # Excluindo a coluna stop_id, pois usamos ela acima somente para fazer a comparação.

display(opcoes_viagem.head())

In [ ]:
opcoes_viagem = opcoes_viagem.merge( # Pegando informações da tabela de rotas e relacionando com a tabela opções de viagem.
    routes[["route_id", "route_short_name", "route_long_name"]], # Selecionando quais são as colunas que quero olhar da tabela rotas.
    on="route_id", # Usando como referencia a coluna route_id para conseguir relacionar as informações das tabelas.
    how="left" # Mantendo as informações da tabela da esquerda.(opções_viagem)
)

opcoes_viagem = opcoes_viagem.rename(columns={ # Renomeando colunas.
    "route_short_name": "numero_linha",
    "route_long_name": "nome_linha"
})

display(opcoes_viagem.head())

In [ ]:
opcoes_viagem["distancia_total"] = ( # Criando uma nova coluna que irá guardar a soma das distâncias de origem e destino.
    opcoes_viagem["distancia_origem"] + opcoes_viagem["distancia_destino"] # Somando as informalções das colunas.
)

opcoes_viagem = opcoes_viagem.sort_values( # Ordenando de forma crescente a tabela, tendo como referencia a coluna distancia_total. Priorizando as viagens que possuem menor distância total.
    by="distancia_total"
)

display(opcoes_viagem.head(10))

## 8. Seleção da viagem e construção do itinerário
    
    Nesta etapa, organizo as opções de viagem pela distância total de caminhada, seleciono a melhor opção encontrada e recupero as informações necessárias para apresentar o percurso ao usuário.

Tomei como foco focar no retorno da melhor viagem/linha para o usuário, mostrar trajetos que o onibus irá fazer e retornar informações importantes para o melhor caminha retornado.

In [ ]:
# CORREÇÃO: antes de pegar a primeira linha, confiro se a tabela tem alguma viagem.
# Se não tiver, o notebook avisa e segue (as próximas células conferem a variável ha_viagem).
if opcoes_viagem.empty:
    ha_viagem = False
    opcao_selecionada = None
    print("Nenhuma viagem direta encontrada entre a origem e o destino (raio de busca de", raio_busca, "m).")
    print("Sugestões: aumentar o raio_busca, trocar a origem/destino ou considerar uma viagem com baldeação.")
else:
    ha_viagem = True
    opcao_selecionada = opcoes_viagem.iloc[0] # Retornando a primeira linha da tabela opcoes_viagem, que é a viagem com menor distância total.

    display(opcao_selecionada) # retornando informações da viagem.

## 8.1 Recuperação do itinerário

Recupero as paradas pertencentes à viagem selecionada e filtro só aquelas que estão entre o ponto de embarque e o ponto de desembarque.

In [ ]:
if not ha_viagem:
    print("Sem viagem recomendada: itinerário não montado.")
else:
    paradas_itinerario = gtfs_completo[ # Copiando as informações da tabela gtfs que são iguais ao trip_id da viagem selecionada.
        gtfs_completo["trip_id"] == opcao_selecionada["trip_id"]
    ].copy()

    paradas_itinerario["stop_sequence"] = pd.to_numeric( # Nesse bloco estou convertendo a coluna para numero.
        paradas_itinerario["stop_sequence"]
    )

    paradas_itinerario = paradas_itinerario[ # Filtrando as informações da tabela para trazer apenas as paradas que estão entre a origem e o destino da viagem selecionada.
        (paradas_itinerario["stop_sequence"] >= opcao_selecionada["stop_sequence_origem"]) &
        (paradas_itinerario["stop_sequence"] <= opcao_selecionada["stop_sequence_destino"])
    ]

    paradas_itinerario = paradas_itinerario.sort_values( # Ordenando as informações da tabela de forma ascendente, sendo "stop_sequence" a coluna de referência.
        by="stop_sequence"
    )

    display( # Estou retornando apenas as colunas que quero analisar.
        paradas_itinerario[
            ["stop_sequence", "stop_id", "stop_name", "stop_lat", "stop_lon"]
        ]
    )

## 8.2 Identificação do sentido da viagem

Consulto as informações da viagem selecionada para identificar seu sentido e o destino indicado no itinerário.

In [ ]:
if not ha_viagem:
    print("Sem viagem recomendada: sentido não identificado.")
else:
    dados_viagem = trips[ # Estou selecionando as informações da tabela trips que são iguais as informações da opcao_selecionada.
        trips["trip_id"] == opcao_selecionada["trip_id"]
    ][["trip_id", "direction_id", "trip_headsign"]].drop_duplicates() #Selecionando as colunas que quero guardar e Removendo informações duplicadas.

    direction_id = dados_viagem.iloc[0]["direction_id"] # Guardo o sentido do GTFS (0 ou 1) para usar nas próximas etapas.

    display(dados_viagem)

## 9. Integração com a API Olho Vivo
    Nesta etapa, uso as informações obtidas no GTFS para realizar consultas à API Olho Vivo e verificar como os dados da viagem podem ser relacionados aos registros que foram dados pela API.



## 9.1 Validação da parada de embarque na API

In [ ]:
if not ha_viagem:
    print("Sem viagem recomendada: nada para validar na API.")
else:
    # Recuperando as coordenadas das paradas de embarque e desembarque no GTFS
    parada_embarque_gtfs = stops[
        stops["stop_id"] == opcao_selecionada["stop_id_origem"]
    ].iloc[0]

    parada_desembarque_gtfs = stops[
        stops["stop_id"] == opcao_selecionada["stop_id_destino"]
    ].iloc[0]

    # Buscando a parada de embarque na API Olho Vivo pelo nome
    # Atenção: erros de conexão/HTTP e respostas vazias agora são tratados pela função consultar_api.
    dados, erro = consultar_api("Parada/Buscar", {"termosBusca": opcao_selecionada["parada_embarque"]})

    if erro:
        print("Não foi possível buscar a parada na API:", erro)
    elif not isinstance(dados, list) or len(dados) == 0:
        print("A API não encontrou paradas com esse nome.")
        print("(O nome do GTFS pode ser escrito diferente do nome da API: a seção 9.3 faz a correspondência pela localização.)")
    else:
        # Padronizando os nomes das colunas da API
        paradas_api_busca = pd.DataFrame(dados).rename(columns={
            "cp": "codigo_parada",
            "np": "nome_parada",
            "ed": "endereco",
            "py": "latitude",
            "px": "longitude"
        })

        colunas_esperadas = {"codigo_parada", "nome_parada", "endereco", "latitude", "longitude"}
        if not colunas_esperadas.issubset(paradas_api_busca.columns):
            print("A API devolveu as paradas em um formato inesperado.")
        else:
            # Calculando a distância entre a parada GTFS e cada resultado da API
            paradas_api_busca["distancia_m"] = paradas_api_busca.apply(
                lambda parada: distancia_m(
                    float(parada_embarque_gtfs["stop_lat"]),
                    float(parada_embarque_gtfs["stop_lon"]),
                    parada["latitude"],
                    parada["longitude"]
                ),
                axis=1
            )

            paradas_api_busca = paradas_api_busca.sort_values(
                by="distancia_m"
            )

            display(
                paradas_api_busca[
                    ["codigo_parada", "nome_parada", "endereco", "distancia_m"]
                ]
            )

## 9.2 Identificação da linha na API

A API pode devolver **várias linhas** para a mesma busca (números parecidos e os dois sentidos da mesma linha). Por isso, comparo o letreiro do GTFS (`route_short_name`, por exemplo `8000-10`) com o letreiro montado a partir da API (`lt` + `-` + `tl`), depois de padronizar espaços e maiúsculas. Só as linhas que realmente coincidem ficam em `linhas_api`.

Se nenhuma coincidir, `linhas_api` fica vazia e o notebook avisa, em vez de usar um código qualquer.

In [ ]:
# 2 Identificação da linha na API.

# linhas_api sempre existe vazia por padrão e só recebe linhas cujo letreiro é igual ao do GTFS.
linhas_api = pd.DataFrame(columns=["cl", "lc", "lt", "sl", "tl", "tp", "ts", "letreiro_api"])

if not ha_viagem:
    print("Sem viagem recomendada: linha não procurada na API.")
else:
    letreiro_gtfs = str(opcao_selecionada["numero_linha"]).strip().upper() # Ex: "8000-10"
    base_letreiro = letreiro_gtfs.split("-")[0]                           # Ex: "8000"

    # Primeiro busco pelo letreiro completo; se não achar, busco só pela primeira parte.
    for termo in [letreiro_gtfs, base_letreiro]:
        dados, erro = consultar_api("Linha/Buscar", {"termosBusca": termo})

        if erro:
            print("Não foi possível buscar a linha na API:", erro)
            break
        if not isinstance(dados, list) or len(dados) == 0:
            continue

        busca = pd.DataFrame(dados)
        if not {"cl", "lt", "tl", "sl"}.issubset(busca.columns):
            print("A API devolveu as linhas em um formato inesperado.")
            break

        # Montando o letreiro da API no mesmo formato do GTFS e comparando (sem assumir que são iguais).
        busca["letreiro_api"] = busca["lt"].astype(str).str.strip().str.upper() + "-" + busca["tl"].astype(str).str.strip()
        encontradas = busca[busca["letreiro_api"] == letreiro_gtfs]

        if not encontradas.empty:
            linhas_api = encontradas.reset_index(drop=True)
            break

    if linhas_api.empty:
        print(f"Nenhuma linha da API tem o letreiro {letreiro_gtfs}. A previsão ficará indisponível.")
    else:
        print(f"Linhas da API com o letreiro {letreiro_gtfs}: {len(linhas_api)} (normalmente 1 por sentido)")
        display(linhas_api[["cl", "letreiro_api", "sl", "tp", "ts"]])

## 9.3 Correspondência entre GTFS e API (linha, sentido e parada)

**Problema:** o GTFS e a API Olho Vivo são bases diferentes. Não posso assumir totalmente que o `stop_id` do GTFS seja o `codigoParada` da API. Isso é **testado** na célula abaixo, não assumido.

**Solução:** para cada sentido candidato da linha (`linhas_api`), busco as paradas dele na API (`Parada/BuscarParadasPorLinha`) e procuro a mais próxima, em metros, do embarque e do desembarque do GTFS (reaproveito a função `distancia_m`). O sentido escolhido é o que deixa as duas paradas mais perto.

A correspondência só é aceita se as distâncias ficarem dentro de `TOLERANCIA_PARADA_M` (50 m foi uma escolha minha; dá para ajustar). Se não ficar, `codigo_linha_api` e `codigo_parada_api` continuam `None` e a previsão fica indisponível. Nada é inventado.

In [ ]:
# Distância máxima em metros para aceitar que uma parada do GTFS e uma parada da API são o mesmo ponto físico.
TOLERANCIA_PARADA_M = 50 # Difinindo o limite máximo de Parada

# Resultado da correspondência continuam None se não for possível validar
codigo_linha_api = None
codigo_parada_api = None
codigo_parada_desembarque_api = None

if not ha_viagem:
    print("Sem viagem recomendada: nada para corresponder.")
elif linhas_api.empty: # Verificando se a linha retornou vazia (empty)
    print("Sem linha correspondente na API: não é possível validar linha e parada.")
else: # Convertendo as variáveis para tipo numérico
    lat_emb = float(parada_embarque_gtfs["stop_lat"])
    lon_emb = float(parada_embarque_gtfs["stop_lon"])
    lat_des = float(parada_desembarque_gtfs["stop_lat"])
    lon_des = float(parada_desembarque_gtfs["stop_lon"])

    candidatas = [] # Uma linha por sentido da linha na API

    for _, linha in linhas_api.iterrows():
        dados, erro = consultar_api("Parada/BuscarParadasPorLinha", {"codigoLinha": int(linha["cl"])})

        if erro:
            print(f"Linha {linha['cl']}: não foi possível buscar as paradas ({erro})")
            continue
        if not isinstance(dados, list) or len(dados) == 0:
            print(f"Linha {linha['cl']}: a API não devolveu paradas.")
            continue

        paradas_da_linha = pd.DataFrame(dados)
        if not {"cp", "py", "px"}.issubset(paradas_da_linha.columns):
            print(f"Linha {linha['cl']}: paradas em formato inesperado.")
            continue

        paradas_da_linha = paradas_da_linha.dropna(subset=["py", "px"])
        if paradas_da_linha.empty:
            print(f"Linha {linha['cl']}: paradas sem coordenadas.")
            continue

        # Distância de cada parada da API até o embarque e até o desembarque do GTFS
        paradas_da_linha["dist_embarque"] = paradas_da_linha.apply(
            lambda p: distancia_m(lat_emb, lon_emb, float(p["py"]), float(p["px"])), axis=1
        )
        paradas_da_linha["dist_desembarque"] = paradas_da_linha.apply(
            lambda p: distancia_m(lat_des, lon_des, float(p["py"]), float(p["px"])), axis=1
        )

        perto_emb = paradas_da_linha.loc[paradas_da_linha["dist_embarque"].idxmin()]
        perto_des = paradas_da_linha.loc[paradas_da_linha["dist_desembarque"].idxmin()]

        candidatas.append({
            "codigo_linha": int(linha["cl"]),
            "sentido_api": linha["sl"],
            "codigo_parada_embarque": perto_emb["cp"],
            "dist_embarque": perto_emb["dist_embarque"],
            "codigo_parada_desembarque": perto_des["cp"],
            "dist_desembarque": perto_des["dist_desembarque"],
            # Teste da suposição "stop_id do GTFS = código da parada na API"
            "stop_id_esta_na_api": (paradas_da_linha["cp"].astype(str) == str(opcao_selecionada["stop_id_origem"])).any()
        })

    if len(candidatas) == 0:
        print("Não foi possível obter as paradas de nenhuma linha candidata: correspondência não validada.")
    else:
        tabela_candidatas = pd.DataFrame(candidatas)
        tabela_candidatas["dist_total"] = tabela_candidatas["dist_embarque"] + tabela_candidatas["dist_desembarque"]
        tabela_candidatas = tabela_candidatas.sort_values("dist_total").reset_index(drop=True)
        display(tabela_candidatas)

        melhor = tabela_candidatas.iloc[0]

        if melhor["dist_embarque"] <= TOLERANCIA_PARADA_M and melhor["dist_desembarque"] <= TOLERANCIA_PARADA_M:
            codigo_linha_api = int(melhor["codigo_linha"])
            codigo_parada_api = int(melhor["codigo_parada_embarque"])
            codigo_parada_desembarque_api = int(melhor["codigo_parada_desembarque"])

            linha_escolhida = linhas_api[linhas_api["cl"].astype(int) == codigo_linha_api].iloc[0]

            print("Correspondência validada pela localização das paradas:")
            print(f"- Linha na API: código {codigo_linha_api} (letreiro {linha_escolhida['letreiro_api']}, sentido da API = {linha_escolhida['sl']})")
            print(f"- Embarque: stop_id GTFS {opcao_selecionada['stop_id_origem']} <-> código da API {codigo_parada_api} (a {melhor['dist_embarque']:.0f} m)")
            print(f"- Desembarque: stop_id GTFS {opcao_selecionada['stop_id_destino']} <-> código da API {codigo_parada_desembarque_api} (a {melhor['dist_desembarque']:.0f} m)")
            print("- O stop_id do GTFS aparece como código de parada na API?", bool(melhor["stop_id_esta_na_api"]))
            print(f"- Sentido no GTFS: direction_id {direction_id}, destino '{dados_viagem.iloc[0]['trip_headsign']}'")
            print(f"- Terminais da API: principal '{linha_escolhida['tp']}', secundário '{linha_escolhida['ts']}'")
            print("  (O sentido foi escolhido pela localização das paradas, não por convenção. Confira se o destino bate com o terminal.)")

            # Se o outro sentido estiver quase tão perto quanto o escolhido, aviso.
            if len(tabela_candidatas) > 1 and (tabela_candidatas.iloc[1]["dist_total"] - melhor["dist_total"]) < 15:
                print("ATENÇÃO: o outro sentido também tem paradas muito próximas. Confira o sentido antes de confiar na previsão.")
        else:
            print("Correspondência NÃO validada: a parada da API mais próxima fica longe demais do GTFS.")
            print(f"Embarque a {melhor['dist_embarque']:.0f} m e desembarque a {melhor['dist_desembarque']:.0f} m (limite: {TOLERANCIA_PARADA_M} m).")
            print("Não vou usar um código de parada não validado: a previsão ficará indisponível.")

## 9.4 Consulta de previsões da linha na API

Agora a consulta usa o `codigo_linha_api` validado na seção 9.3 (antes era um código fixo). Também confere se a parada validada aparece entre as paradas com previsão. Se ela não aparecer, isso não é necessariamente um erro: a API só lista paradas que têm ônibus previstos no momento.

In [ ]:
# Previsão da linha. CORREÇÃO: usa o codigo_linha_api validado na seção 9.3, e não mais um código fixo.

if not ha_viagem:
    print("Sem viagem recomendada: previsão da linha não consultada.")
elif codigo_linha_api is None:
    print("Sem linha validada na API (seção 9.3): previsão da linha não consultada.")
else:
    dados, erro = consultar_api("Previsao/Linha", {"codigoLinha": codigo_linha_api})

    if erro:
        print("Não foi possível consultar a previsão da linha:", erro)
    elif not isinstance(dados, dict) or not dados.get("ps"):
        print("A API não retornou previsões para esta linha agora (pode não haver ônibus em operação).")
    else:
        pontos_previsao = pd.DataFrame(dados["ps"])

        if not {"cp", "np", "py", "px", "vs"}.issubset(pontos_previsao.columns):
            print("A API devolveu a previsão em um formato inesperado.")
        else:
            pontos_previsao["distancia_gtfs"] = pontos_previsao.apply(
                lambda parada: distancia_m(
                    float(parada_embarque_gtfs["stop_lat"]),
                    float(parada_embarque_gtfs["stop_lon"]),
                    float(parada["py"]),
                    float(parada["px"])
                ),
                axis=1
            )

            pontos_previsao = pontos_previsao.sort_values(
                by="distancia_gtfs"
            )

            display(
                pontos_previsao[
                    ["cp", "np", "py", "px", "distancia_gtfs", "vs"]
                ]
            )

            if codigo_parada_api is not None:
                aparece = (pontos_previsao["cp"].astype(str) == str(codigo_parada_api)).any()
                print("A parada de embarque validada aparece entre as paradas com previsão agora?", aparece)

## 10. Análises exploratórias e testes da API

        Organizei as células que usei para investigar os recursos da API e entender os dados que ela disponibiliza.

**Observação:** as células 5 e 9 abaixo usam códigos fixos como exemplos de teste. Esses códigos **não** alimentam a recomendação; a recomendação usa apenas `codigo_linha_api` e `codigo_parada_api`, localizados na seção 9.3.

In [ ]:
# 3 Investigar as paradas retornadas para cada linha.

codigos_linha = linhas_api["cl"].tolist()

for codigo in codigos_linha:
    dados, erro = consultar_api("Parada/BuscarParadasPorLinha", {"codigoLinha": int(codigo)})

    print(f"\nCódigo da linha: {codigo}")
    if erro:
        print("Não foi possível consultar:", erro)
    elif not dados:
        print("A API não retornou paradas.")
    else:
        display(pd.DataFrame(dados).head())

In [ ]:
# 4 Verificar as previsões disponíveis para várias linhas.
for codigo in linhas_api["cl"]:
    dados_previsao, erro = consultar_api("Previsao/Linha", {"codigoLinha": int(codigo)})

    print(f"\nCódigo da linha: {codigo}")
    if erro:
        print("Não foi possível consultar:", erro)
    elif not isinstance(dados_previsao, dict):
        print("Resposta em formato inesperado.")
    else:
        print("Horário de referência:", dados_previsao.get("hr"))
        print("Pontos retornados:", len(dados_previsao.get("ps") or []))

In [ ]:
# 5 Testar a consulta de previsão de uma linha específica (32892).
# Código fixo usado só como EXEMPLO de teste: não faz parte da recomendação.
dados_previsao, erro = consultar_api("Previsao/Linha", {"codigoLinha": 32892})

if erro:
    print("Não foi possível consultar:", erro)
elif not isinstance(dados_previsao, dict) or not dados_previsao.get("ps"):
    print("Sem previsões para esta linha agora.")
else:
    pontos_previsao = pd.DataFrame(dados_previsao["ps"])
    display(pontos_previsao)

In [ ]:
# 9 Consulta de previsão da parada, após a reformulação.
# Código fixo usado só como EXEMPLO de teste: não faz parte da recomendação.
dados_previsao_parada, erro = consultar_api("Previsao/Parada", {"codigoParada": 70015949})

if erro:
    print("Não foi possível consultar:", erro)
else:
    print(dados_previsao_parada)

# 11. Mapa interativo e quadro da recomendação

Nesta etapa, construo o mapa pedido pelo professor com a biblioteca Folium e um quadro com o resultado da recomendação, incluindo a previsão de chegada real da API Olho Vivo.

Tudo aqui reaproveita o que já existe no notebook: `origem`, `destino`, `opcao_selecionada`, `paradas_itinerario`, `dados_viagem`, `trips`, `sessao` e a função `distancia_m`. A lógica da recomendação (escolha da viagem direta) não foi alterada.

**Principais correções desta versão** (marcadas com `CORREÇÃO` nos comentários):
- Se não houver viagem direta, o notebook avisa e segue sem erro (`ha_viagem`).
- A previsão usa o código de parada e de linha **da API**, validados pela localização (seção 9.3), e não o `stop_id` do GTFS.
- O trajeto só usa o `shapes.txt` se ele existir, tiver pontos e passar perto das paradas; senão, o notebook diz isso e usa a ligação entre paradas, identificada como aproximação.
- O mapa usa o provedor de tiles CartoDB Positron (o mapa de fundo padrão do Folium é o do OpenStreetMap, que bloqueia o carregamento dentro do Colab).
- Todas as consultas à API tratam erros e respostas vazias (`consultar_api`).

## 11.1 Trajeto da viagem

**Objetivo:** ter a lista de coordenadas que o ônibus percorre entre o embarque e o desembarque, para desenhar no mapa.

**Dados usados:** o `shapes.txt` do GTFS (desenho oficial da linha), ligado à viagem pelo `shape_id` da tabela `trips`.

**Como funciona:** acho o ponto do shape mais próximo do embarque e o mais próximo do desembarque (procurando o desembarque só depois do embarque) e fico com o pedaço entre eles.

**CORREÇÃO:** antes de usar o shape, a célula confere se (1) a coluna `shape_id` e o arquivo `shapes.txt` existem, (2) o `shape_id` da viagem tem pelo menos 2 pontos no arquivo, (3) o shape passa a menos de `TOLERANCIA_SHAPE_M` metros das duas paradas e (4) o desembarque vem depois do embarque. Se algum teste falhar, a célula mostra o motivo e usa a sequência das paradas (`paradas_itinerario`), que também é dado real, mas liga as paradas em linha reta. Esse trajeto é identificado como **aproximação**, não como o caminho real pelas ruas.

In [ ]:
import folium

# Distância máxima (em metros) entre uma parada e o ponto mais próximo do shape para eu considerar o shape compatível.
TOLERANCIA_SHAPE_M = 100

def ponto_shape_mais_proximo(tabela_shape, lat, lon):
    """Devolve o índice do ponto do shape mais próximo de (lat, lon) e a distância até ele, em metros."""
    distancias = tabela_shape.apply(
        lambda ponto: distancia_m(lat, lon, ponto["shape_pt_lat"], ponto["shape_pt_lon"]), axis=1
    )
    return distancias.idxmin(), distancias.min()

# Valores padrão, para as próximas células funcionarem mesmo sem viagem recomendada
trajeto = []
tipo_trajeto = "indisponível"
trajeto_e_oficial = False

if not ha_viagem:
    print("Sem viagem recomendada: trajeto não montado.")
else:
    # Coordenadas das paradas do itinerário (já vêm do GTFS)
    paradas_itinerario = paradas_itinerario.copy()
    paradas_itinerario["stop_lat"] = pd.to_numeric(paradas_itinerario["stop_lat"])
    paradas_itinerario["stop_lon"] = pd.to_numeric(paradas_itinerario["stop_lon"])

    embarque = paradas_itinerario.iloc[0]
    desembarque = paradas_itinerario.iloc[-1]

    # Tentando usar o shape oficial da viagem. Se algo falhar, guardo o motivo em motivo_sem_shape.
    motivo_sem_shape = ""
    arquivo_shapes = f"{PASTA_GTFS}/shapes.txt"

    if "shape_id" not in trips.columns:
        motivo_sem_shape = "o trips.txt não tem a coluna shape_id."
    elif not os.path.exists(arquivo_shapes):
        motivo_sem_shape = "o arquivo shapes.txt não existe na pasta do GTFS."
    else:
        shape_id = trips.loc[trips["trip_id"] == opcao_selecionada["trip_id"], "shape_id"].iloc[0]

        if pd.isna(shape_id):
            motivo_sem_shape = "a viagem não tem shape_id."
        else:
            shapes = pd.read_csv(arquivo_shapes, dtype=str)
            shape_viagem = shapes[shapes["shape_id"] == shape_id].copy()

            # CORREÇÃO: só continuo se o shape_id realmente tem coordenadas no shapes.txt
            for coluna in ["shape_pt_sequence", "shape_pt_lat", "shape_pt_lon"]:
                shape_viagem[coluna] = pd.to_numeric(shape_viagem[coluna], errors="coerce")
            shape_viagem = shape_viagem.dropna(subset=["shape_pt_sequence", "shape_pt_lat", "shape_pt_lon"])

            if len(shape_viagem) < 2:
                motivo_sem_shape = f"o shape_id {shape_id} tem {len(shape_viagem)} ponto(s) válido(s) no shapes.txt (preciso de pelo menos 2)."
            else:
                shape_viagem = shape_viagem.sort_values("shape_pt_sequence").reset_index(drop=True)

                inicio, dist_inicio = ponto_shape_mais_proximo(shape_viagem, embarque["stop_lat"], embarque["stop_lon"])
                restante = shape_viagem.loc[inicio:]
                fim, dist_fim = ponto_shape_mais_proximo(restante, desembarque["stop_lat"], desembarque["stop_lon"])

                if dist_inicio > TOLERANCIA_SHAPE_M or dist_fim > TOLERANCIA_SHAPE_M:
                    motivo_sem_shape = f"o shape passa a {dist_inicio:.0f} m do embarque e a {dist_fim:.0f} m do desembarque (limite: {TOLERANCIA_SHAPE_M} m)."
                elif fim <= inicio:
                    motivo_sem_shape = "no shape, o desembarque não vem depois do embarque."
                else:
                    trecho = shape_viagem.loc[inicio:fim]
                    trajeto = trecho[["shape_pt_lat", "shape_pt_lon"]].values.tolist()
                    tipo_trajeto = "shape oficial do GTFS"
                    trajeto_e_oficial = True

    # Alternativa: sequência das paradas da viagem (dado real, mas em linhas retas entre as paradas)
    if not trajeto_e_oficial:
        print("Shape oficial não usado:", motivo_sem_shape)
        trajeto = paradas_itinerario[["stop_lat", "stop_lon"]].values.tolist()
        tipo_trajeto = "ligação em linha reta entre as paradas da viagem (aproximação: NÃO é o traçado real pelas ruas)"

    print("Trajeto montado com:", tipo_trajeto)
    print("Pontos no trajeto:", len(trajeto))

## 11.2 Previsão de chegada

**Objetivo:** buscar na API quando o próximo ônibus da linha recomendada chega ao ponto de embarque.

**Dados usados:** o endpoint `Previsao/Parada` da Olho Vivo.

**CORREÇÃO:** a consulta agora usa `codigo_parada_api` e `codigo_linha_api`, validados pela localização na seção 9.3. Antes o código usava o `stop_id` do GTFS como se fosse o código da API, e comparava o letreiro como texto. Agora a linha é escolhida pelo **código da linha da API** (`cl`), que já identifica o sentido, e o letreiro devolvido pela API (`c`) é mostrado no quadro para conferência.

**Como funciona:** a função devolve `(previsao, motivo)`. Se a API não responder, não tiver a linha ou não tiver ônibus previstos, `previsao` é `None` e `motivo` explica o que aconteceu, e o resto da recomendação continua sendo mostrado. Nada é inventado. A espera em minutos é calculada com o horário de consulta da API (`hr`), o que também funciona perto da meia-noite.

In [ ]:
def buscar_previsao(codigo_parada, codigo_linha):
    """Consulta Previsao/Parada. Devolve (previsao, motivo): se não houver previsão, previsao é None e motivo explica por quê."""
    dados, erro = consultar_api("Previsao/Parada", {"codigoParada": int(codigo_parada)})
    if erro:
        return None, erro

    if not isinstance(dados, dict):
        return None, "a API devolveu a previsão em formato inesperado."

    parada = dados.get("p")
    if not parada:
        return None, "a API não tem previsões para este ponto agora."

    # Escolho a linha pelo código da API (cl), que já identifica a linha e o sentido.
    linhas_previsao = [l for l in (parada.get("l") or []) if str(l.get("cl")) == str(codigo_linha)]
    if not linhas_previsao:
        return None, "a linha recomendada não aparece nas previsões deste ponto agora."

    linha_api = linhas_previsao[0]
    veiculos = [v for v in (linha_api.get("vs") or []) if v.get("t")]
    if not veiculos:
        return None, "a linha está listada, mas sem ônibus com previsão no momento."

    # CORREÇÃO: calculo a espera em minutos a partir do horário de consulta, para ordenar certo mesmo perto da meia-noite.
    hora_consulta = dados.get("hr")
    try:
        agora = int(hora_consulta[:2]) * 60 + int(hora_consulta[3:5])
        for v in veiculos:
            v["espera_min"] = (int(v["t"][:2]) * 60 + int(v["t"][3:5]) - agora) % 1440
    except (TypeError, ValueError):
        return None, "a API devolveu horários em formato inesperado."

    veiculos = sorted(veiculos, key=lambda v: v["espera_min"])

    return {
        "horario_consulta": hora_consulta,
        "proximo_onibus": veiculos[0]["t"],
        "espera_min": veiculos[0]["espera_min"],
        "prefixo": veiculos[0].get("p"),
        "onibus_previstos": len(veiculos),
        "letreiro_api": linha_api.get("c"),
        "destino_api": linha_api.get("lt0")
    }, None

previsao = None

if not ha_viagem:
    motivo_sem_previsao = "não há viagem recomendada."
elif codigo_parada_api is None or codigo_linha_api is None:
    motivo_sem_previsao = "não foi possível validar a correspondência entre o GTFS e a API (seção 9.3)."
else:
    previsao, motivo_sem_previsao = buscar_previsao(codigo_parada_api, codigo_linha_api)

print(previsao if previsao else f"Previsão indisponível: {motivo_sem_previsao}")

## 11.3 Mapa interativo

**Objetivo:** mostrar no mapa a origem, o destino, o embarque, o desembarque, as paradas intermediárias e o trajeto do ônibus.

**Por que usei Folium:** o follium gera mapas interativos (zoom, clique nos marcadores) que aparecem direto no Colab, achei melhor pois não precisa de servidor.

**O erro `Access blocked`:** investiguei antes de trocar o provedor. O mapa de fundo padrão do Folium vem dos servidores de tiles do OpenStreetMap, eles pedem o cabeçalho `Referer` em cada requisição (é uma política de uso de tiles).

Dentro do Colab, o mapa é mostrado em um quadro isolado (iframe) e o navegador pode não enviar esse cabeçalho, então o OpenStreetMap recusa os tiles e mostra "Access blocked". Isso afeta **só o mapa de fundo**: marcadores e linhas são desenhados pelo navegador e continuam aparecendo (sobre fundo cinza). Por isso troquei o provedor para o **CartoDB Positron**, que não tem essa exigência.

Usei a célula seguinte para testar se o endereço dos tiles responde a partir do Colab. Atenção: quem baixa os tiles é o navegador, não o Colab, ou seja esse teste mostra que o servidor está no ar, mas **a confirmação final é visual**: o mapa deve mostrar as ruas ao fundo.

**Como funciona:** criei o mapa centralizado na região da viagem, adicionei um marcador para cada elemento (cores e ícones diferentes, com um popup mostrando nome, código e função na viagem), numero as paradas intermediárias na ordem da viagem e desenho o trajeto com `PolyLine`. Uma legenda explica as cores. As linhas tracejadas cinza ligam origem ao embarque e desembarque até o destino e representam só a caminhada aproximada, não o trajeto do ônibus.

In [ ]:
# Provedores de tiles (mapa de fundo). Nenhum deles exige chave de API.
## O CartoDB Positron passou a pedir chave ("API KEY REQUIRED"), então troquei por Esri (padrão) + OpenStreetMap (opcional).

TILES_ESRI_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}" # Esri usa a ordem z/y/x
TILES_ESRI_ATTR = "Tiles &copy; Esri"
TILES_OSM_URL = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"
TILES_OSM_ATTR = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'

# Teste: os endereços dos tiles respondem a partir do Colab? (baixei 1 tile da região da origem)

from math import log, tan, pi

zoom = 13
n = 2 ** zoom
tile_x = int((origem["longitude"] + 180) / 360 * n) # Converte longitude para o número do tile
lat_rad = radians(origem["latitude"])
tile_y = int((1 - log(tan(lat_rad) + 1 / cos(lat_rad)) / pi) / 2 * n) # Converte latitude para o número do tile

for nome, modelo in [("Esri", TILES_ESRI_URL), ("OpenStreetMap", TILES_OSM_URL)]:
    url_teste = modelo.format(z=zoom, x=tile_x, y=tile_y)
    try:
        resposta_tile = requests.get(url_teste, timeout=10, headers={"User-Agent": "Projeto-Transporte-academico"})
        print(nome, "-> HTTP", resposta_tile.status_code, "| tipo:", resposta_tile.headers.get("Content-Type"))
    except requests.exceptions.RequestException as erro:
        print(nome, "-> sem acesso a partir do Colab:", type(erro).__name__)

In [ ]:
def texto_popup(titulo, nome, codigo, funcao):
    return folium.Popup(
        f"<b>{titulo}</b><br>Nome: {nome}<br>Código: {codigo}<br>Função: {funcao}",
        max_width=300
    )

mapa = None # Fica None se não tiver viagem recomendada

if not ha_viagem:
    print("Sem viagem recomendada: mapa não gerado.")
else:
    centro = [(origem["latitude"] + destino["latitude"]) / 2, (origem["longitude"] + destino["longitude"]) / 2]

    # CORREÇÃO: tiles=None tira o fundo padrão (OpenStreetMap sem Referer, que gera "Access blocked").
    # Esri é o fundo padrão; o OpenStreetMap fica disponível no botão de camadas, já com o referrerPolicy recomendado.
    mapa = folium.Map(location=centro, zoom_start=13, tiles=None)
    folium.TileLayer(tiles=TILES_ESRI_URL, attr=TILES_ESRI_ATTR, name="Esri World Street Map", max_zoom=19).add_to(mapa)
    folium.TileLayer(
        tiles=TILES_OSM_URL, attr=TILES_OSM_ATTR, name="OpenStreetMap", max_zoom=19, show=False,
        referrerPolicy="strict-origin-when-cross-origin"
    ).add_to(mapa)
    folium.LayerControl().add_to(mapa)

    # Origem e destino informados
    folium.Marker(
        [origem["latitude"], origem["longitude"]],
        popup=folium.Popup(f"<b>Origem</b><br>Lat/Lon: {origem['latitude']}, {origem['longitude']}", max_width=300),
        tooltip="Origem",
        icon=folium.Icon(color="green", icon="play", prefix="fa")
    ).add_to(mapa)

    folium.Marker(
        [destino["latitude"], destino["longitude"]],
        popup=folium.Popup(f"<b>Destino</b><br>Lat/Lon: {destino['latitude']}, {destino['longitude']}", max_width=300),
        tooltip="Destino",
        icon=folium.Icon(color="red", icon="flag", prefix="fa")
    ).add_to(mapa)

    # Embarque e desembarque
    folium.Marker(
        [embarque["stop_lat"], embarque["stop_lon"]],
        popup=texto_popup("Ponto de embarque", embarque["stop_name"], embarque["stop_id"], "onde o passageiro entra no ônibus"),
        tooltip="Embarque",
        icon=folium.Icon(color="blue", icon="bus", prefix="fa")
    ).add_to(mapa)

    folium.Marker(
        [desembarque["stop_lat"], desembarque["stop_lon"]],
        popup=texto_popup("Ponto de desembarque", desembarque["stop_name"], desembarque["stop_id"], "onde o passageiro desce do ônibus"),
        tooltip="Desembarque",
        icon=folium.Icon(color="purple", icon="bus", prefix="fa")
    ).add_to(mapa)

    # Paradas intermediárias (entre o embarque e o desembarque), numeradas na ordem da viagem
    for numero, (_, parada) in enumerate(paradas_itinerario.iloc[1:-1].iterrows(), start=1):
        folium.CircleMarker(
            [parada["stop_lat"], parada["stop_lon"]],
            radius=5,
            color="orange",
            fill=True,
            fill_opacity=0.9,
            popup=texto_popup(f"Parada intermediária nº {numero}", parada["stop_name"], parada["stop_id"], f"parada nº {parada['stop_sequence']} da viagem"),
            tooltip=f"{numero}. {parada['stop_name']}"
        ).add_to(mapa)

    # Trajeto do ônibus: linha cheia se for o shape oficial; tracejada se for só a ligação entre paradas
    if trajeto_e_oficial:
        folium.PolyLine(trajeto, color="blue", weight=4, opacity=0.8, tooltip=f"Linha {opcao_selecionada['numero_linha']} (shape oficial)").add_to(mapa)
    else:
        folium.PolyLine(trajeto, color="blue", weight=3, opacity=0.8, dash_array="2 8", tooltip="Ligação em linha reta entre paradas (aproximação)").add_to(mapa)

    # Caminhada aproximada até o embarque e depois do desembarque
    folium.PolyLine(
        [[origem["latitude"], origem["longitude"]], [embarque["stop_lat"], embarque["stop_lon"]]],
        color="gray", weight=3, dash_array="6", tooltip="Caminhada aproximada (linha reta)"
    ).add_to(mapa)

    folium.PolyLine(
        [[desembarque["stop_lat"], desembarque["stop_lon"]], [destino["latitude"], destino["longitude"]]],
        color="gray", weight=3, dash_array="6", tooltip="Caminhada aproximada (linha reta)"
    ).add_to(mapa)

    # Legenda para ajudar a interpretar o mapa
    legenda = f"""
    <div style="position: fixed; bottom: 25px; left: 25px; z-index: 9999; background: white; padding: 8px 12px;
                border: 1px solid #888; border-radius: 6px; font-size: 12px; line-height: 1.6; max-width: 280px;">
      <b>Linha {opcao_selecionada['numero_linha']}</b><br>
      <span style="color: green;">&#9679;</span> Origem informada<br>
      <span style="color: red;">&#9679;</span> Destino informado<br>
      <span style="color: #2a81cb;">&#9679;</span> Embarque (ponto de ônibus)<br>
      <span style="color: purple;">&#9679;</span> Desembarque (ponto de ônibus)<br>
      <span style="color: orange;">&#9679;</span> Paradas intermediárias (numeradas)<br>
      <span style="color: blue;">&#9473;</span> Trajeto: {tipo_trajeto}<br>
      <span style="color: gray;">&#9473; &#9473;</span> Caminhada aproximada (linha reta)
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(legenda))

    # Ajustando o zoom para caber tudo
    todos_pontos = trajeto + [
        [origem["latitude"], origem["longitude"]],
        [destino["latitude"], destino["longitude"]]
    ]
    mapa.fit_bounds([
        [min(p[0] for p in todos_pontos), min(p[1] for p in todos_pontos)],
        [max(p[0] for p in todos_pontos), max(p[1] for p in todos_pontos)]
    ])

mapa

## 11.4 Informações da recomendação

**Objetivo:** o objetivo aqui é apresentar o resultado da recomendação logo abaixo do mapa.

**Dados usados:** `opcao_selecionada` (linha, pontos e distâncias), `dados_viagem` (sentido) e `previsao` (resultado da API). Calculei as distâncias na seção 7.1 com a fórmula de Haversine, em linha reta.

**Observação:** quando a previsão não está disponível, acrescentei ao algiritmo a função de dizer o motivo; quando está, mostra também o letreiro e o destino que a API informou, para você conferir se é mesmo a linha e o sentido recomendados.

In [ ]:
if not ha_viagem:
    print("RECOMENDAÇÃO DE VIAGEM")
    print("-" * 60)
    print("Nenhuma viagem direta encontrada entre a origem e o destino.")
    print("Sugestões: aumentar o raio_busca ou trocar a origem/destino.")
else:
    if previsao:
        texto_previsao = f"{previsao['proximo_onibus']} (em cerca de {previsao['espera_min']} min; veículo {previsao['prefixo']}, consulta às {previsao['horario_consulta']})"
    else:
        texto_previsao = f"indisponível ({motivo_sem_previsao})"

    print("RECOMENDAÇÃO DE VIAGEM")
    print("-" * 60)
    print("Linha recomendada:", opcao_selecionada["numero_linha"], "-", opcao_selecionada["nome_linha"])
    print("Sentido:", dados_viagem.iloc[0]["trip_headsign"], f"(direction_id {direction_id})")
    print("Ponto de embarque:", opcao_selecionada["parada_embarque"], f"(stop_id GTFS {opcao_selecionada['stop_id_origem']}; código na API: {codigo_parada_api if codigo_parada_api is not None else 'não validado'})")
    print("Previsão do próximo ônibus:", texto_previsao)
    if previsao:
        print(f"  Conferência com a API: letreiro {previsao['letreiro_api']}, destino '{previsao['destino_api']}'")
    print("Ponto de desembarque:", opcao_selecionada["parada_desembarque"], f"(stop_id GTFS {opcao_selecionada['stop_id_destino']})")
    print("Paradas intermediárias:", max(len(paradas_itinerario) - 2, 0))
    print(f"Distância origem -> embarque: {opcao_selecionada['distancia_origem']:.0f} m (em linha reta)")
    print(f"Distância desembarque -> destino: {opcao_selecionada['distancia_destino']:.0f} m (em linha reta)")
    print("-" * 60)
    print("Trajeto desenhado com:", tipo_trajeto)

## 11.5 Verificações da recomendação

Resumo do que foi validado nesta execução. O que está com ATENÇÃO, pode não ser necessariamente um erro: significa que o notebook usou uma alternativa ou que o dado não estava disponível para o retorno (veja a mensagem da célula correspondente).

In [ ]:
verificacoes = [
    ("1. Existem viagens diretas candidatas", ha_viagem),
    ("2. Parada do GTFS corresponde a uma parada da API (validada pela localização)", codigo_parada_api is not None),
    ("3. O shapes.txt tem o traçado desta viagem (senão, usa a ligação entre paradas)", trajeto_e_oficial),
    ("4. A linha foi identificada na API (letreiro igual ao do GTFS)", codigo_linha_api is not None),
    ("5. A API devolveu uma previsão válida", previsao is not None),
    ("6. O mapa Folium foi gerado", mapa is not None),
]

for descricao, ok in verificacoes:
    print("OK      " if ok else "ATENÇÃO ", descricao)

if ha_viagem:
    ordem_ok = paradas_itinerario["stop_sequence"].is_monotonic_increasing
    print("OK      " if ordem_ok else "ATENÇÃO ", "   Paradas do itinerário em ordem crescente de stop_sequence")

print("MANUAL  ", "7. O mapa aparece no Colab com as ruas ao fundo? Se aparecer só fundo cinza com marcadores, o problema é o provedor de tiles, não os dados.")